# 03 — Leakage-Safe Feature Engineering

This notebook builds the modeling table for next-basket reorder prediction. Each row represents a `user_id` + `product_id` candidate pair, where features are computed only from historical prior orders and labels are derived from the user's train order.

The goal is to simulate the real business problem: given a customer's past grocery behavior, predict which previously purchased products they will reorder in their next basket.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
base_path = "/Volumes/workspace/default/instacart"  # change if needed
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"

dbutils.fs.mkdirs(gold_path)

In [0]:
prior = spark.read.parquet(f"{silver_path}/prior_order_details")
train = spark.read.parquet(f"{silver_path}/train_order_details")

## 1. Using prior orders for features, train orders for labels

In [0]:
prior_features_base = prior
train_labels_base = train

### Leakage prevention design

Features are computed only from `prior` orders. Labels are computed only from the user's `train` order. No product, user, or user-product feature uses the target train basket, which prevents the model from learning the answer directly.

## 2. Creating candidate user-product pairs

Only users with a `train` order are used for supervised modeling, because users in the `test` split do not have observable target basket labels. Candidate products are restricted to items the user has purchased historically.

Users belonging to the test evaluation set do not have observable target basket labels. For supervised reorder prediction, candidate user-product pairs were restricted to users with a train order so that each row has valid target-order context and a known reorder label.

In [0]:
train_users = train_labels_base.select("user_id").distinct()

candidates = (
    prior_features_base
    .select("user_id", "product_id")
    .distinct()
    .join(train_users, on="user_id", how="inner")
)

print("Candidate user-product pairs for train users:", candidates.count())

## 3. Creating labels from train order

The label is binary: `1` if the candidate product appears in the user's train order, and `0` otherwise. This converts next-basket prediction into a supervised ranking/classification problem.

In [0]:
labels = (
    train_labels_base
    .select("user_id", "product_id")
    .distinct()
    .withColumn("label", F.lit(1))
)

In [0]:
training_base = (
    candidates
    .join(labels, on=["user_id", "product_id"], how="left")
    .fillna({"label": 0})
)

display(training_base.limit(20))

In [0]:
display(
    training_base.groupBy("label")
    .agg(F.count("*").alias("rows"))
)

## 4. User-level features

In [0]:
user_features = (
    prior_features_base
    .groupBy("user_id")
    .agg(
        F.countDistinct("order_id").alias("user_total_orders"),
        F.count("*").alias("user_total_items"),
        F.countDistinct("product_id").alias("user_unique_products"),
        F.avg("reordered").alias("user_reorder_rate"),
        F.avg("days_since_prior_order").alias("user_avg_days_between_orders"),
        F.avg("add_to_cart_order").alias("user_avg_cart_position")
    )
    .withColumn(
        "user_avg_basket_size",
        F.col("user_total_items") / F.col("user_total_orders")
    )
)

display(user_features.limit(10))

## 5. Product-level features

Product reorder rates are smoothed using the global reorder rate to reduce noise from low-volume products. This prevents rarely purchased products with artificially high raw reorder rates from dominating the model.

In [0]:
global_reorder_rate = prior_features_base.agg(F.avg("reordered")).first()[0]
smoothing_factor = 100

In [0]:
product_features = (
    prior_features_base
    .groupBy("product_id")
    .agg(
        F.count("*").alias("product_total_purchases"),
        F.sum("reordered").alias("product_total_reorders"),
        F.avg("reordered").alias("product_raw_reorder_rate"),
        F.avg("add_to_cart_order").alias("product_avg_cart_position"),
        F.countDistinct("user_id").alias("product_unique_users"),
        F.first("aisle_id").alias("aisle_id"),
        F.first("department_id").alias("department_id")
    )
    .withColumn(
        "product_smoothed_reorder_rate",
        (F.col("product_total_reorders") + F.lit(global_reorder_rate * smoothing_factor)) /
        (F.col("product_total_purchases") + F.lit(smoothing_factor))
    )
)

display(product_features.limit(10))

## 6. User-product interaction features

User-product interaction features are expected to be the strongest predictors because grocery reorder behavior is highly personalized. These features capture how often a specific user bought a specific product, how recently they bought it, and how consistently it was reordered.

In [0]:
user_product_features = (
    prior_features_base
    .groupBy("user_id", "product_id")
    .agg(
        F.count("*").alias("up_times_purchased"),
        F.sum("reordered").alias("up_times_reordered"),
        F.avg("reordered").alias("up_reorder_rate"),
        F.avg("add_to_cart_order").alias("up_avg_cart_position"),
        F.min("order_number").alias("up_first_order_number"),
        F.max("order_number").alias("up_last_order_number")
    )
)

display(user_product_features.limit(10))

## 7. User latest order context

In [0]:
user_latest_prior_order = (
    prior_features_base
    .select(
        "user_id",
        "order_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    )
    .dropDuplicates(["user_id", "order_id"])
)

In [0]:
w_latest = Window.partitionBy("user_id").orderBy(F.desc("order_number"))

user_latest_order_features = (
    user_latest_prior_order
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumnRenamed("order_number", "user_last_order_number")
    .withColumnRenamed("order_dow", "last_order_dow")
    .withColumnRenamed("order_hour_of_day", "last_order_hour")
    .withColumnRenamed("days_since_prior_order", "last_days_since_prior_order")
    .drop("order_id")
)

display(user_latest_order_features.limit(10))

## 8. Deriving recency features

In [0]:
user_product_features = (
    user_product_features
    .join(user_latest_order_features.select("user_id", "user_last_order_number"), on="user_id", how="left")
    .withColumn(
        "up_orders_since_last_purchase",
        F.col("user_last_order_number") - F.col("up_last_order_number")
    )
    .withColumn(
        "up_purchase_frequency",
        F.col("up_times_purchased") / F.col("user_last_order_number")
    )
)

## 9. Training the order context

In [0]:
train_order_context = (
    train_labels_base
    .select(
        "user_id",
        "order_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    )
    .dropDuplicates(["user_id", "order_id"])
    .withColumnRenamed("order_id", "target_order_id")
    .withColumnRenamed("order_number", "target_order_number")
    .withColumnRenamed("order_dow", "target_order_dow")
    .withColumnRenamed("order_hour_of_day", "target_order_hour")
    .withColumnRenamed("days_since_prior_order", "target_days_since_prior_order")
)

display(train_order_context.limit(10))

## 10. Assemblying full training set

In [0]:
training_set = (
    training_base
    .join(user_features, on="user_id", how="left")
    .join(product_features, on="product_id", how="left")
    .join(user_product_features, on=["user_id", "product_id"], how="left")
    .join(user_latest_order_features, on="user_id", how="left")
    .join(train_order_context, on="user_id", how="left")
)

In [0]:
numeric_fill_values = {
    "user_reorder_rate": 0.0,
    "user_avg_days_between_orders": 0.0,
    "user_avg_cart_position": 0.0,
    "user_avg_basket_size": 0.0,
    "product_raw_reorder_rate": 0.0,
    "product_smoothed_reorder_rate": 0.0,
    "product_avg_cart_position": 0.0,
    "up_reorder_rate": 0.0,
    "up_avg_cart_position": 0.0,
    "up_orders_since_last_purchase": 999.0,
    "up_purchase_frequency": 0.0,
    "last_days_since_prior_order": 0.0,
    "target_days_since_prior_order": 0.0
}

training_set = training_set.fillna(numeric_fill_values)

## 11. Selecting final columns

In [0]:
final_training_set = training_set.select(
    "user_id",
    "product_id",
    "target_order_id",
    "label",

    # user features
    "user_total_orders",
    "user_total_items",
    "user_unique_products",
    "user_reorder_rate",
    "user_avg_days_between_orders",
    "user_avg_cart_position",
    "user_avg_basket_size",

    # product features
    "product_total_purchases",
    "product_total_reorders",
    "product_raw_reorder_rate",
    "product_smoothed_reorder_rate",
    "product_avg_cart_position",
    "product_unique_users",
    "aisle_id",
    "department_id",

    # user-product features
    "up_times_purchased",
    "up_times_reordered",
    "up_reorder_rate",
    "up_avg_cart_position",
    "up_first_order_number",
    "up_last_order_number",
    "up_orders_since_last_purchase",
    "up_purchase_frequency",

    # order context
    "last_order_dow",
    "last_order_hour",
    "last_days_since_prior_order",
    "target_order_dow",
    "target_order_hour",
    "target_days_since_prior_order"
)

In [0]:
display(final_training_set.limit(20))

## 12. Validating the feature table

In [0]:
print("Rows:", final_training_set.count())
print("Users:", final_training_set.select("user_id").distinct().count())
print("Products:", final_training_set.select("product_id").distinct().count())

In [0]:
display(
    final_training_set.groupBy("label")
    .agg(F.count("*").alias("rows"))
)

In [0]:
missing_summary = final_training_set.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in final_training_set.columns
])

display(missing_summary)

In [0]:
display(
    final_training_set.select(
        F.count(F.when(F.col("target_order_id").isNull(), 1)).alias("missing_target_order_id"),
        F.count(F.when(F.col("target_order_dow").isNull(), 1)).alias("missing_target_order_dow"),
        F.count(F.when(F.col("target_order_hour").isNull(), 1)).alias("missing_target_order_hour")
    )
)

Target-order context is checked separately because missing target fields usually indicate that test-set users were accidentally included in the supervised training table.

## 13. Saving the gold training table

In [0]:
final_training_set.write.mode("overwrite").parquet(f"{gold_path}/reorder_training_set")

In [0]:
reorder_training_set = spark.read.parquet(f"{gold_path}/reorder_training_set")
display(reorder_training_set.limit(10))

In [0]:
feature_groups = spark.createDataFrame([
    ("user", "user_total_orders", "Number of historical orders placed by the user"),
    ("user", "user_reorder_rate", "User's historical item-level reorder rate"),
    ("user", "user_avg_basket_size", "Average number of items per historical order"),
    ("product", "product_smoothed_reorder_rate", "Product reorder rate adjusted with global smoothing"),
    ("product", "product_unique_users", "Number of users who purchased the product historically"),
    ("user_product", "up_times_purchased", "Number of times the user bought the product historically"),
    ("user_product", "up_orders_since_last_purchase", "Number of orders since the user last bought the product"),
    ("user_product", "up_purchase_frequency", "Share of user's orders containing the product"),
    ("order_context", "target_days_since_prior_order", "Days since the user's previous order at target time")
], ["feature_group", "feature_name", "description"])

display(feature_groups)

feature_groups.write.mode("overwrite").parquet(f"{gold_path}/feature_dictionary")

## Feature Engineering Summary

This notebook creates a leakage-safe training dataset for next-basket reorder prediction. The modeling unit is a user-product candidate pair, where candidate products are limited to products the user has purchased historically.

Features are computed only from prior orders, while labels are derived from the user's train order. This prevents target leakage and better simulates the real business problem: predicting which previously purchased products a user will reorder in their next basket.

Feature groups include:

- User-level shopping behavior
- Product-level popularity and reorder behavior
- User-product interaction history
- Recency and purchase frequency features
- Target order timing context

Product reorder rates are smoothed to avoid overvaluing low-volume products with artificially high reorder percentages.